In [1]:
import pandas as pd
import numpy as np


In [2]:
# Read in RASMI material intensity data and MI from old IMAGE-Materials CP
mi_rasmi = pd.read_excel("MI_ranges_20230905.xlsx", index_col=[0, 1, 2, 3, 4], 
                         sheet_name=["concrete", "brick", "wood", "steel", "glass", "plastics", "aluminum", "copper"])

mi_image_mat = pd.read_csv("Building_materials_old.csv", index_col=[0, 1, 2])
# remove all data
mi_image_mat.loc[:, :] = np.nan

mi_image_mat_commercial = pd.read_csv("materials_commercial_old.csv", index_col=[0, 1, 2])


In [3]:
def expand_image_mat_mi(mi_image_mat: pd.DataFrame,):
    # also get 2030 data in there
    # Get all unique index values for Region and HousingType
    regions = mi_image_mat.index.get_level_values(1).unique()
    housing_types = mi_image_mat.index.get_level_values(2).unique()

    # Create new index tuples for 2030
    new_index = pd.MultiIndex.from_product([[2030], regions, housing_types], names=mi_image_mat.index.names)

    # Add missing 2030 rows if not present
    mi_image_mat = mi_image_mat.reindex(mi_image_mat.index.union(new_index))

    # Now you can safely assign values for 2030
    mi_image_mat.loc[(2030, slice(None), slice(None)), :] = mi_image_mat.loc[(2020, slice(None), slice(None)), :].values
    return mi_image_mat

In [4]:
# dictionary that maps RASMI R32 Regions to IMAGE R26 Regions

image_to_rasmi = {
    1: 'OECD_CAN',
    2: 'OECD_USA',
    3: 'LAM_MEX',
    4: 'LAM_LAM-L',
    5: 'LAM_BRA',
    6: 'LAM_LAM-L',
    7: ['MAF_MEA-H', 'MAF_MEA-M', 'MAF_NAF'],
    8: ['MAF_SSA-L', 'MAF_SSA-M'],
    9: ['MAF_SSA-L', 'MAF_SSA-M'],
    10: 'MAF_SAF',
    11: ['OECD_EFTA', 'OECD_EU12-H', 'OECD_EU12-M', 'OECD_EU15'],
    12: 'REF_EEU-FSU',
    13: 'OECD_TUR',
    14: ['REF_EEU-FSU', 'ASIA_TWN'],
    15: ['OECD_EEU', 'REF_CAS'],
    16: 'REF_RUS',
    17: ['MAF_MEA-H', 'MAF_MEA-M'],
    18: ['ASIA_IND', 'ASIA_OAS-CPA', 'ASIA_PAK'],
    19: 'OECD_KOR',
    20: 'ASIA_CHN',
    21: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    22: 'ASIA_IDN',
    23: 'OECD_JPN',
    24: 'OECD_AUNZ',
    25: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    26: ['MAF_SSA-L', 'MAF_SSA-M']
}

housing_type_image_to_rasmi = {
    1 : 'RS', #detached house to residential single family
    2 : 'RS', # semi detached house to residential single family
    3 : 'RM', # appartement to residential multi-family
    4 : 'RM', # high-rise to residential multi-family
}

housing_type_rasmi_to_image = {
    'RS' : [1, 2], # detached house and semi detached house to
    'RM' : [3, 4]  # appartement and high-rise to
}

housing_type_to_rasmi_building_structure = {
    1 : ['C', 'M', 'S', 'T'], # assumption that detached housing are average all structures
    2 : ['C', 'M', 'S', 'T'], # assumption that semi detached housing are average all structures
    3 : ['C', 'S'], # assumption that appartement are only made out of cement and steel structures
    4 : ['S'] # assumption that high-rise only steel structures
}

material_list_rasmi = ["steel", "concrete", "wood", "copper", "aluminum", "glass", "brick"]
mis_list_target = ["steel", "concrete", "wood", "copper", "aluminium", "glass", "brick"]

# regions where total steel consumption is higher than what is estimated by image-materials sectors buildings & vehicles
steel_regions_adapt = [4, 8, 9, 22, 25, 26]

In [5]:
def replace_old_mis_with_rasmi(mi_rasmi: pd.DataFrame, material_list_rasmi: list, start_year: int = 2020, 
                               target_year: int = 2050, data_value = "p_50"):
    """    
    Replace old material intensities in IMAGE-Materials with RASMI values for concrete.
    
    material_name: str, non capitalised from rasmi
    mis_list_target_name: str, material intensity in IMAGE-Materials
    data_value: str, can also be 0, 25, 75 or 100 percentile
    """
    # standardize column names at the top of the function
    mi_image_mat.columns = [c.lower().replace("aluminum", "aluminium") for c in mi_image_mat.columns]
    
    for material_name in material_list_rasmi:
        # ensure lower case
        if material_name.lower() == "aluminum":
            material_name_image = "aluminium"
        else:
            material_name_image = material_name.lower()

        print(material_name_image, material_name)
        material_intensities = mi_rasmi.get(material_name)

        # loop through all IMAGE classes and the according rasmi regions
        for image_region, rasmi_region in image_to_rasmi.items():
            # convert str to list is necessary:
            if isinstance(rasmi_region, str):
                rasmi_region = [rasmi_region]
            
            # loop through IMAGE regions and get the mean concrete mi value for each region for RS and RM (housing types)
            for housingtype_image, housingtype_rasmi in housing_type_image_to_rasmi.items():
                    
                if image_region in steel_regions_adapt and material_name == "steel":
                    data_value = "p_5"
                else:
                    data_value = "p_50"

                filtered_mis = material_intensities[material_intensities.index.get_level_values('R5_32').isin(rasmi_region) #filter for the right region
                                        & material_intensities.index.get_level_values('function').isin([housingtype_rasmi]) # filter for the right housing type of rasmi
                                        & material_intensities.index.get_level_values('structure').isin(housing_type_to_rasmi_building_structure[housingtype_image])].loc[:, data_value] # filter for the right building structure

                mean_mi_value = filtered_mis.mean()
                # replace old values with new values
                mi_image_mat.loc[([start_year, target_year], image_region, housingtype_image), material_name_image] = mean_mi_value
                # save as csv
    mi_image_mat.to_csv("Building_materials_rasmi.csv")
    print("done")
    return mi_image_mat




In [6]:
# execute the function to replace old mis with rasmi values
replace_old_mis_with_rasmi(mi_rasmi, material_list_rasmi)

steel steel
concrete concrete
wood wood
copper copper
aluminium aluminum
glass glass
brick brick
done


steel    concrete       wood    copper  \
Year Region Building_type                                               
2020 1      1              43.303105  574.250973  50.041437  0.182895   
            2              43.303105  574.250973  50.041437  0.182895   
            3              55.383413  762.544862  20.600000  0.182895   
            4              70.500000  534.983854  22.000000  0.182895   
     2      1              42.327828  481.434722  48.768730  0.182895   
...                              ...         ...        ...       ...   
2050 25     4               7.267620  734.855156  22.000000  0.182895   
     26     1               7.303744  572.352907  50.201717  0.182895   
            2               7.303744  572.352907  50.201717  0.182895   
            3              10.948612  821.157646  20.600000  0.182895   
            4               3.822224  677.167328  22.000000  0.182895   

                           aluminium     glass       brick  
Year Region Building_type                                   
2020 1      1               0.489213  3.023166  244.713738  
            2               0.489213  3.023166  244.713738  
            3               0.491529  2.825000  139.333333  
            4               0.490000  2.650000   32.666667  
     2      1               0.490000  4.005891  240.375000  
...                              ...       ...         ...  
2050 25     4               0.490000  2.000000  107.000000  
     26     1               0.490765  3.206462  271.712209  
            2               0.490765  3.206462  271.712209  
            3               0.490000  2.234286  159.795751  
            4               0.490000  2.000000  107.000000  

[208 rows x 7 columns]

In [7]:


def replace_old_mis_with_rasmi_resource_efficient(mi_image_mat: pd.DataFrame,
                                                  mi_rasmi: pd.DataFrame,
                                                  material_list_rasmi: list, 
                                                  start_year: int = 2020, 
                                                  switch_year: int = 2030, 
                                                  target_year: int = 2050, 
                                                  data_value = "p_50"):
    """    
    Replace old material intensities in IMAGE-Materials with RASMI values for concrete.
    
    material_name: str, non capitalised from rasmi
    mis_list_target_name: str, name of the material intensity in IMAGE-Materials
    data_value: str, can also be 0, 25, 75 or 100 percentile
    """
    # standardize column names at the top of the function
    mi_image_mat.columns = [c.lower().replace("aluminum", "aluminium") for c in mi_image_mat.columns]

    for material_name in material_list_rasmi:
        # ensure lower case
        if material_name.lower() == "aluminum":
            material_name_image = "aluminium"
        else:
            material_name_image = material_name.lower()

        print(material_name_image, material_name)
        material_intensities = mi_rasmi.get(material_name)

        # loop through all years
        for year in [start_year, switch_year, target_year]:
            print(year)
            if year == start_year:
                data_value = "p_50"
            if year == switch_year:
                data_value = "p_50"
            if year == target_year:
                data_value = "p_25"
            # loop through all IMAGE classes and the according rasmi regions
            for image_region, rasmi_region in image_to_rasmi.items():
                # convert str to list is necessary:
                if isinstance(rasmi_region, str):
                    rasmi_region = [rasmi_region]
                
                # loop through IMAGE regions and get the mean concrete mi value for each region for RS and RM (housing types)
                for housingtype_image, housingtype_rasmi in housing_type_image_to_rasmi.items():
                        
                    if image_region in steel_regions_adapt and material_name == "steel":
                        data_value = "p_5"
                    else:
                        if year in [start_year] and image_region not in steel_regions_adapt and material_name != "steel":
                            print('switch back')
                            data_value = "p_50"
                        if year in [switch_year] and image_region not in steel_regions_adapt and material_name != "steel":
                            print('switch to more efficient')
                            data_value = "p_25"

                    filtered_mis = material_intensities[material_intensities.index.get_level_values('R5_32').isin(rasmi_region) #filter for the right region
                                            & material_intensities.index.get_level_values('function').isin([housingtype_rasmi]) # filter for the right housing type of rasmi
                                            & material_intensities.index.get_level_values('structure').isin(housing_type_to_rasmi_building_structure[housingtype_image])].loc[:, data_value] # filter for the right building structure

                    mean_mi_value = filtered_mis.mean()
                    # replace old values with new values
                    mi_image_mat.loc[([year], image_region, housingtype_image), material_name_image] = mean_mi_value

                
                # save as csv
    mi_image_mat.to_csv("Building_materials_rasmi_resource_efficient.csv")
    print("done")
    return mi_image_mat




In [8]:
mi_image_mat = expand_image_mat_mi(mi_image_mat)
test = replace_old_mis_with_rasmi_resource_efficient(mi_image_mat, mi_rasmi, material_list_rasmi)

steel steel
2020
2030
2050
concrete concrete
2020
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
switch back
sw

In [9]:
# test.loc[(2020, 1, 1), :] - 
test.loc[(2030, 1, 1), :] - test.loc[(2050, 1, 1), :]


steel        19.878604
concrete      0.000000
wood          0.000000
copper        0.000000
aluminium     0.000000
glass         0.000000
brick         0.000000
dtype: float64